In [ ]:
import os
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import nltk

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

nltk.download("punkt", quiet=True)

# Helper function to load data from standard format
def load_dataset_files(file_paths):
    english_sentences = []
    hinglish_sentences = []
    for path in file_paths:
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        english_sentences.append(parts[0])
                        hinglish_sentences.append(parts[1])
    return pd.DataFrame({"english": english_sentences, "hinglish": hinglish_sentences})

file_paths = ["/content/train.txt", "/content/dev.txt", "/content/test.txt"]
df = load_dataset_files(file_paths)

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["english"] = df["english"].apply(clean_text)
df["hinglish"] = df["hinglish"].apply(clean_text)

df = df[
    (df["english"].str.len() > 0) &
    (df["hinglish"].str.len() > 0)
]
df = df.drop_duplicates().reset_index(drop=True)

df["hinglish"] = df["hinglish"].apply(
    lambda s: "startseq " + s + " endseq"
)

MAX_SAMPLES = min(len(df), 20000)
data = df.sample(
    n=MAX_SAMPLES, random_state=42
).reset_index(drop=True)

english_tokenizer = Tokenizer(oov_token="<unk>", filters="")
hinglish_tokenizer = Tokenizer(oov_token="<unk>", filters="")

english_tokenizer.fit_on_texts(data["english"])
hinglish_tokenizer.fit_on_texts(data["hinglish"])

english_sequences = english_tokenizer.texts_to_sequences(data["english"])
hinglish_sequences = hinglish_tokenizer.texts_to_sequences(data["hinglish"])

MAX_ENGLISH_LEN = min(max(map(len, english_sequences)), 40)
MAX_HINGLISH_LEN = min(max(map(len, hinglish_sequences)), 42)

english_sequences = pad_sequences(
    english_sequences, maxlen=MAX_ENGLISH_LEN, padding="post", truncating="post"
)
hinglish_sequences = pad_sequences(
    hinglish_sequences, maxlen=MAX_HINGLISH_LEN, padding="post", truncating="post"
)

english_vocab_size = len(english_tokenizer.word_index) + 1
hinglish_vocab_size = len(hinglish_tokenizer.word_index) + 1

X_train, X_test, Y_train, Y_test = train_test_split(
    english_sequences, hinglish_sequences, test_size=0.15, random_state=42
)

Y_train_input = Y_train[:, :-1]
Y_train_output = Y_train[:, 1:]
Y_test_input = Y_test[:, :-1]
Y_test_output = Y_test[:, 1:]

sample_weights = (Y_train_output != 0).astype("float32")

# Fast tf.data input pipeline
BATCH_SIZE = 128
train_dataset = tf.data.Dataset.from_tensor_slices((
    (X_train, Y_train_input), Y_train_output, sample_weights
))
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

EMBEDDING_DIM = 128
LATENT_DIM = 256

encoder_inputs = Input(shape=(MAX_ENGLISH_LEN,), name="encoder_inputs")
encoder_embedding = Embedding(english_vocab_size, EMBEDDING_DIM, mask_zero=True)(encoder_inputs)

encoder_lstm = LSTM(LATENT_DIM, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(MAX_HINGLISH_LEN - 1,), name="decoder_inputs")
decoder_embedding_layer = Embedding(
    hinglish_vocab_size, EMBEDDING_DIM, mask_zero=True, name="decoder_embedding"
)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm"
)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(hinglish_vocab_size, activation="softmax", name="output_layer")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["sparse_categorical_accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)

# Train on fast tf.data pipeline
history = model.fit(
    train_dataset,
    validation_data=((X_test, Y_test_input), Y_test_output, (Y_test_output != 0).astype("float32")),
    epochs=10,
    callbacks=[early_stop],
    verbose=1
)

encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_h = Input(shape=(LATENT_DIM,))
decoder_state_c = Input(shape=(LATENT_DIM,))
decoder_token = Input(shape=(1,))

decoder_emb = decoder_embedding_layer(decoder_token)
decoder_out, next_h, next_c = decoder_lstm(
    decoder_emb, initial_state=[decoder_state_h, decoder_state_c]
)
decoder_out = decoder_dense(decoder_out)

decoder_model = Model(
    [decoder_token, decoder_state_h, decoder_state_c],
    [decoder_out, next_h, next_c]
)

index_to_word = {index: word for word, index in hinglish_tokenizer.word_index.items()}

def translate(sentence):
    sentence = clean_text(sentence)
    seq = english_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=MAX_ENGLISH_LEN, padding="post", truncating="post")

    states = encoder_model.predict(seq, verbose=0)
    start_id = hinglish_tokenizer.word_index.get("startseq")
    end_id = hinglish_tokenizer.word_index.get("endseq")

    if start_id is None or end_id is None:
        return ""

    token = np.array([[start_id]])
    result = []

    for _ in range(MAX_HINGLISH_LEN - 1):
        output, h, c = decoder_model.predict([token, states[0], states[1]], verbose=0)
        predicted_id = int(np.argmax(output[0, -1, :]))

        if predicted_id == end_id or predicted_id == 0:
            break

        word = index_to_word.get(predicted_id, "<unk>")
        if word not in ["startseq", "endseq"]:
            result.append(word)

        token = np.array([[predicted_id]])
        states = [h, c]

    return " ".join(result)

smoothie = SmoothingFunction().method1
references = []
predictions = []

for i in range(min(len(X_test), 50)):
    reference_ids = Y_test[i]
    reference_words = [
        index_to_word.get(int(t), "")
        for t in reference_ids
        if t != 0 and int(t) not in [
            hinglish_tokenizer.word_index.get("startseq"),
            hinglish_tokenizer.word_index.get("endseq")
        ]
    ]
    english_words = [english_tokenizer.index_word.get(int(t), "") for t in X_test[i] if t != 0]
    predicted = translate(" ".join(english_words))
    predictions.append(predicted.split())
    references.append([reference_words])

bleu = corpus_bleu(references, predictions, smoothing_function=smoothie)
print("\nTest BLEU score:", round(bleu, 4))

unseen_sentences = [
    "How are you today?",
    "I am going to school.",
    "Please help me with this problem.",
    "What are you doing right now?",
    "I want to learn something new."
]

print("\nUNSEEN ENGLISH TO HINGLISH TRANSLATIONS")
for sentence in unseen_sentences:
    print("English :", sentence)
    print("Hinglish:", translate(sentence))
    print("-" * 50)

Epoch 1/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 9s 122ms/step - loss: 6.6083 - sparse_categorical_accuracy: 0.0694 - val_loss: 5.8659 - val_sparse_categorical_accuracy: 0.0859
Epoch 2/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 115ms/step - loss: 5.7397 - sparse_categorical_accuracy: 0.0944 - val_loss: 5.6928 - val_sparse_categorical_accuracy: 0.0942
Epoch 3/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 116ms/step - loss: 5.5493 - sparse_categorical_accuracy: 0.1017 - val_loss: 5.5632 - val_sparse_categorical_accuracy: 0.1041
Epoch 4/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 116ms/step - loss: 5.4064 - sparse_categorical_accuracy: 0.1090 - val_loss: 5.4745 - val_sparse_categorical_accuracy: 0.1157
Epoch 5/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 7s 120ms/step - loss: 5.2739 - sparse_categorical_accuracy: 0.1222 - val_loss: 5.3812 - val_sparse_categorical_accuracy: 0.1234
Epoch 6/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 7s 122ms/step - loss: 5.1446 - sparse_categorical_accuracy: 0.1312 - val_loss: 5.3001 - val_sparse_categorical_accuracy: 0.1319
Epoc